# 01 · Run one complete model day

Choose one `MODEL_ID`, review the derived status, then explicitly enable the
expensive stage. Rerunning all cells resumes only compatible unfinished work.
Search rungs, final checkpoints, evaluation, profiling, and reports are
discovered automatically from persistent storage.


In [ ]:
MODEL_ID = "faster_rcnn_resnet50"
BENCHMARK_TRACK = "controlled"

RUN_MODE = "auto"
RUN_LR_RANGE_TEST = True
RUN_BOUNDARY_EXTENSION = False

START_EXPENSIVE_STAGE = False
ALLOW_OVER_BUDGET_RUN = False
DATA_ACCESS_MODE = "local_cache"


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git"
REPOSITORY_BRANCH = "main"
SMOKE_TEST = os.environ.get("SMOKE_TEST", "").lower() in {"1", "true", "yes"}

# The only logic a notebook still owns: make `src` importable. Everything after
# this line - Git state, platform detection, paths, dependency policy - lives in
# src/notebook_bootstrap.py so all notebooks behave identically.
_override = os.environ.get("BENCHMARK_REPO_ROOT")
_candidates = (
    [Path(_override).expanduser()]
    if _override
    else [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/content/aerial-object-detection-benchmark"),
        Path("/kaggle/working/aerial-object-detection-benchmark"),
    ]
)
REPO_PATH = next(
    (
        candidate.resolve()
        for candidate in _candidates
        if (candidate / "src" / "notebook_bootstrap.py").is_file()
    ),
    None,
)
if REPO_PATH is None:
    _host = (
        Path("/content")
        if Path("/content").is_dir()
        else Path("/kaggle/working")
        if Path("/kaggle/working").is_dir()
        else None
    )
    if _host is None:
        raise RuntimeError(
            "Run this notebook from the repository, or set BENCHMARK_REPO_ROOT "
            "to an existing clone."
        )
    REPO_PATH = (_host / "aerial-object-detection-benchmark").resolve()
    REPO_PATH.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_PATH)],
        check=True,
    )
sys.path.insert(0, str(REPO_PATH))

from src.notebook_bootstrap import bootstrap_notebook

bootstrap = bootstrap_notebook(
    REPO_PATH,
    requirements_file='requirements-dataset-colab.txt',
    use_google_drive=True,
    smoke_test=SMOKE_TEST,
)
notebook_environment = bootstrap.environment
REPO_PATH = notebook_environment.repository_root
DRIVE_ROOT = notebook_environment.artifact_root
LOCAL_CACHE_ROOT = notebook_environment.local_cache_root
NOTEBOOK_PLATFORM = notebook_environment.platform
IN_COLAB = NOTEBOOK_PLATFORM == "colab"
IN_KAGGLE = NOTEBOOK_PLATFORM == "kaggle"
print(bootstrap.summary())

RESOLVED_DATA_ACCESS_MODE = (
    DATA_ACCESS_MODE if NOTEBOOK_PLATFORM == "colab" else "drive_direct"
)


In [ ]:
from src.data.contract import verify_complete_data_contract
from src.config.benchmark_tracks import load_track_config, track_output_root
from src.data.local_cache import resolve_data_access
from src.pathing.resolver import resolve_legacy_paths
from src.models.resnet_frcnn.factory import RESNET_FACTORY_MODEL_ID
from src.models.swin_frcnn.factory import SWIN_FACTORY_MODEL_ID
from src.models.vmamba_frcnn.factory import VMAMBA_FACTORY_MODEL_ID
from src.models.rtdetrv2.factory import RTDETR_FACTORY_MODEL_ID
from src.workflows.model_day import inspect_model_day
from src.workflows.notebook_entrypoints import require_verified_data

track_config = load_track_config(REPO_PATH, BENCHMARK_TRACK)
print(f"Benchmark track: {BENCHMARK_TRACK}; outputs: {track_output_root(DRIVE_ROOT, BENCHMARK_TRACK)}")
paths = resolve_legacy_paths(DRIVE_ROOT)
if MODEL_ID == RESNET_FACTORY_MODEL_ID:
    print("ResNet factory: AVAILABLE")
if MODEL_ID == SWIN_FACTORY_MODEL_ID:
    print("Swin factory and feature adapter: AVAILABLE")
if MODEL_ID == VMAMBA_FACTORY_MODEL_ID:
    print("VMamba factory and importer: AVAILABLE")
if MODEL_ID == RTDETR_FACTORY_MODEL_ID:
    print("RT-DETRv2 factory: AVAILABLE")
    print("Parameter-group report: diagnostic global-LR mode")
if SMOKE_TEST:
    from src.data.download import VISDRONE_ARCHIVES
    for spec in VISDRONE_ARCHIVES.values():
        spec["minimum_bytes"] = 1
max_images = 12 if SMOKE_TEST else None
if RESOLVED_DATA_ACCESS_MODE == "local_cache":
    persistent_contract = verify_complete_data_contract(
        paths,
        repo_root=REPO_PATH,
        max_images_per_split=max_images,
    )
    require_verified_data(persistent_contract)
data_access = resolve_data_access(
    paths,
    RESOLVED_DATA_ACCESS_MODE,
    cache_root=LOCAL_CACHE_ROOT,
)
contract = verify_complete_data_contract(
    paths,
    repo_root=REPO_PATH,
    local_cache=data_access if data_access.mode == "local_cache" else None,
    max_images_per_split=max_images,
)
require_verified_data(contract)
print("DATA CONTRACT VERIFIED: YES")
print(json.dumps(contract.to_dict(), indent=2, default=str))

state = inspect_model_day(
    DRIVE_ROOT,
    MODEL_ID,
    REPO_PATH,
    verify_data=False,
)
state["data_contract"] = contract.to_dict()
print(json.dumps(state, indent=2, default=str))
if state["stage"] == "DATA":
    raise RuntimeError(
        "Dataset setup is incomplete. Run 00_prepare_visdrone.ipynb, then rerun this notebook."
    )


In [ ]:
from src.workflows.model_day import ModelDayOptions, run_model_day
result = run_model_day(
    REPO_PATH,
    DRIVE_ROOT,
    ModelDayOptions(
        model_id=MODEL_ID,
        run_mode=RUN_MODE,
        run_lr_range_test=RUN_LR_RANGE_TEST,
        run_boundary_extension=RUN_BOUNDARY_EXTENSION,
        start_expensive_stage=START_EXPENSIVE_STAGE,
        allow_over_budget_run=ALLOW_OVER_BUDGET_RUN,
        smoke_test=SMOKE_TEST,
        data_access_mode=RESOLVED_DATA_ACCESS_MODE,
        local_cache_root=str(LOCAL_CACHE_ROOT),
    ),
    data_access=data_access,
    verified_data_contract=contract.to_dict(),
)
print(json.dumps(result, indent=2, default=str))


In [ ]:
if result["stage"] == "COMPLETE":
    evaluation = json.loads(Path(result["evaluation_paths"][0]).read_text())
    print("MODEL DAY COMPLETE")
    print()
    print(f"Model: {MODEL_ID}")
    print(f"Selected LR: {result['selected_lr']}")
    print(f"Final run ID: {result['final_run_id']}")
    print(f"Best checkpoint: {result['checkpoint_path']}")
    print(f"mAP50-95: {evaluation.get('mAP')}")
    print(f"APtiny: {evaluation.get('APtiny')}")
    print(f"Report: {result['report_path']}")
    print(f"Recommended bundle: {result['recommended_bundle']}")
    print("Next notebook: 02_publish_results.ipynb")
else:
    print(result.get("message", f"Next stage: {result['stage']}"))
